# 获得股票日频数据

In [ ]:
import duckdb
import pandas as pd
from typing import Optional, Iterable
import numpy as np



BASE_PATH = r"D:\database\stock_basic_data_daily"   # 配置：合并后的数据根目录（只保留 year/month 分区）
# BASE_PATH = r"C:\Users\Administrator\Desktop\筹码分布"   # 配置：合并后的数据根目录（只保留 year/month 分区）




VIEW_NAME = "stock_day_merged"

con = duckdb.connect()   # 初始化 DuckDB 连接
con.execute(f"""
CREATE OR REPLACE VIEW {VIEW_NAME} AS
SELECT *
FROM read_parquet('{BASE_PATH}/year=*/month=*/merged.parquet', hive_partitioning=1)
""")    # 创建视图：自动识别 year/month 分区，读取所有 merged.parquet

print("视图创建完成：", VIEW_NAME)
print("数据路径：", BASE_PATH)    

# 查看有哪些年月分区（可选）
# df_partitions = con.execute(f"SELECT DISTINCT year, month FROM {VIEW_NAME} ORDER BY year, month").df(); print("\n现有分区："); print(df_partitions)

In [ ]:
from cgi import print_arguments


target_codes = ['000950.SZ']
codes_str = ", ".join([f"'{code}'" for code in target_codes])

df_multi = con.execute(f"SELECT * FROM {VIEW_NAME} WHERE htsc_code IN ({codes_str})  ORDER BY htsc_code, time").df()
# df_multi[['time','抄底总分']] #and df_multi['time']>'2026-04-01 16:29:18'


# # 这里需要固定时间格式，方便后续的统一时间轴计算
df_multi['time'] = pd.to_datetime(df_multi['time']).dt.strftime('%Y-%m-%d')

df_multi#.tail(10)


In [ ]:
df  = df_multi[df_multi['htsc_code'] == '001376.SZ']



# df.columns
# df.to_csv('D:\database\signal_daily\merged.csv', index=False)
# df

df[['time','high','low']]

# 交易数据（换手，流通，每笔成交额等）
### 股票换手、流通市值，总市值，交易状态，换手率，成交笔数，振幅，平均每笔成交量，平均每笔成交额

数据目录：`D:\database\stock_financial_statements\market_equity_data`（由 `获得股票日频换手率.py` 写入，hive 分区 `year=*/month=*/merged.parquet`）。

包含 `get_daily_basic` 返回的全部字段，例如：

- 行情：`open` / `high` / `low` / `close` / `prev_close` / `volume` / `value`
- 换手：`turnover_rate`（百分数）、`turnover_deals`
- 其它：`day_change`、`amplitude`、`avg_price`、`floating_market_val`、`total_market_val`、`name`、`exchange`、`trading_state` 等

In [ ]:
TEMP_DATA_BASE_PATH = r"D:\database\stock_financial_statements\market_equity_data"
TEMP_DATA_VIEW_NAME = "market_equity_data_merged"

import duckdb
import pandas as pd

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW {TEMP_DATA_VIEW_NAME} AS
SELECT *
FROM read_parquet('{TEMP_DATA_BASE_PATH}/year=*/month=*/merged.parquet', hive_partitioning=1, union_by_name=True)
""")

print("视图创建完成：", TEMP_DATA_VIEW_NAME)
print("数据路径：", TEMP_DATA_BASE_PATH)

schema_df = con.execute(f"DESCRIBE SELECT * FROM {TEMP_DATA_VIEW_NAME}").df()
print("\n字段列表：")
print(schema_df["column_name"].tolist())

# 可选：查看已有 year/month 分区
# con.execute(f"SELECT DISTINCT year, month FROM {TEMP_DATA_VIEW_NAME} ORDER BY year, month").df()

In [ ]:
# 日行情基础库概览：有哪些代码、各自条数与日期范围
con.execute(f"""
SELECT
    UPPER(TRIM(CAST(htsc_code AS VARCHAR))) AS htsc_code,
    COUNT(*) AS row_count,
    MIN(CAST(time AS DATE)) AS first_date,
    MAX(CAST(time AS DATE)) AS last_date
FROM {TEMP_DATA_VIEW_NAME}
GROUP BY 1
ORDER BY htsc_code
""").df().tail(20)

In [ ]:
# 全量获取全部字段（整表读入内存；数据量大时可改用下一 cell 按代码筛选）
df_temp_all = con.execute(f"""
SELECT *
FROM {TEMP_DATA_VIEW_NAME}
ORDER BY htsc_code, time
""").df()

print("总行数:", len(df_temp_all))
print("列名:", list(df_temp_all.columns))

if "time" in df_temp_all.columns:
    tser = pd.to_datetime(df_temp_all["time"], errors="coerce")
    print("time 范围:", tser.min(), "~", tser.max())
    df_temp_all["time"] = tser.dt.strftime("%Y-%m-%d")

if "htsc_code" in df_temp_all.columns:
    print("不重复 htsc_code 数:", df_temp_all["htsc_code"].nunique())

df_temp_all.tail(10)

In [ ]:
# 按股票代码筛选全部字段（改 target_codes / end_date 即可）
target_codes = ["001376.SZ"]
end_date = "2023-11-23"

codes_str = ", ".join([f"'{code}'" for code in target_codes])
df_temp_data = con.execute(f"""
SELECT *
FROM {TEMP_DATA_VIEW_NAME}
WHERE htsc_code IN ({codes_str})
  AND CAST(time AS DATE) <= DATE '{end_date}'
ORDER BY htsc_code, time
""").df()

df_temp_data["time"] = pd.to_datetime(df_temp_data["time"]).dt.strftime("%Y-%m-%d")
print("行数:", len(df_temp_data))
print("列名:", list(df_temp_data.columns))
if not df_temp_data.empty:
    print("代码:", df_temp_data["htsc_code"].unique().tolist())
    print("区间:", df_temp_data["time"].min(), "~", df_temp_data["time"].max())
df_temp_data.tail(15)


df = df_temp_data

df = df[['time','high','low','turnover_rate']]

df.to_csv(r'C:\Users\Administrator\Desktop\python_venv\工具\123.csv')

In [ ]:
df_temp_data[df_temp_data['trading_state'] != '1']

# 市值数据

数据目录：`D:\database\stock_financial_statements\stock_valuation_data`（由 `获得市值数据.py` 写入，hive 分区 `year=*/month=*/merged.parquet`）。

字段（`get_stock_valuation`，`trading_day` 存为 `time`）：

- 标识：`htsc_code`, `exchange`, `time`
- 估值：`pe`, `pettm`, `pb`, `pc`, `pcttm`, `ps`, `psttm`
- 市值：`floating_market_val`, `total_market_val`


In [ ]:
VALUATION_BASE_PATH = r"D:\database\stock_financial_statements\stock_valuation_data"
VALUATION_VIEW_NAME = "stock_valuation_data_merged"

import duckdb
import pandas as pd

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW {VALUATION_VIEW_NAME} AS
SELECT *
FROM read_parquet('{VALUATION_BASE_PATH}/year=*/month=*/merged.parquet', hive_partitioning=1, union_by_name=True)
""")

print("视图创建完成：", VALUATION_VIEW_NAME)
print("数据路径：", VALUATION_BASE_PATH)

schema_df = con.execute(f"DESCRIBE SELECT * FROM {VALUATION_VIEW_NAME}").df()
print("\n字段列表：")
print(schema_df["column_name"].tolist())

# 可选：查看已有 year/month 分区
# con.execute(f"SELECT DISTINCT year, month FROM {VALUATION_VIEW_NAME} ORDER BY year, month").df()

In [ ]:
# 估值库概览：有哪些代码、各自条数与日期范围
con.execute(f"""
SELECT
    UPPER(TRIM(CAST(htsc_code AS VARCHAR))) AS htsc_code,
    COUNT(*) AS row_count,
    MIN(CAST(time AS DATE)) AS first_date,
    MAX(CAST(time AS DATE)) AS last_date
FROM {VALUATION_VIEW_NAME}
GROUP BY 1
ORDER BY htsc_code
""").df().tail(20)

In [ ]:
# 全量概览（数据量大时可跳过，改用下一 cell 按代码筛选）
df_val_all = con.execute(f"""
SELECT *
FROM {VALUATION_VIEW_NAME}
ORDER BY htsc_code, time
""").df()

print("总行数:", len(df_val_all))
print("列名:", list(df_val_all.columns))

if df_val_all.empty:
    print("暂无市值数据，请先运行 工具/获得市值数据.py")
else:
    if "time" in df_val_all.columns:
        tser = pd.to_datetime(df_val_all["time"], errors="coerce")
        print("time 范围:", tser.min(), "~", tser.max())
        df_val_all["time"] = tser.dt.strftime("%Y-%m-%d")
    if "htsc_code" in df_val_all.columns:
        print("不重复 htsc_code 数:", df_val_all["htsc_code"].nunique())
    df_val_all.tail(10)

In [ ]:
# 按股票代码筛选估值字段（改 target_codes / end_date 即可）
target_codes = ["601688.SH", "600519.SH"]
end_date = "2026-05-22"

codes_str = ", ".join([f"'{code}'" for code in target_codes])
df_valuation = con.execute(f"""
SELECT
    htsc_code,
    exchange,
    time,
    pe,
    pettm,
    pb,
    pc,
    pcttm,
    ps,
    psttm,
    floating_market_val,
    total_market_val
FROM {VALUATION_VIEW_NAME}
WHERE htsc_code IN ({codes_str})
  AND CAST(time AS DATE) <= DATE '{end_date}'
ORDER BY htsc_code, time
""").df()

df_valuation["time"] = pd.to_datetime(df_valuation["time"]).dt.strftime("%Y-%m-%d")
print("行数:", len(df_valuation))
if not df_valuation.empty:
    print("代码:", df_valuation["htsc_code"].unique().tolist())
    print("区间:", df_valuation["time"].min(), "~", df_valuation["time"].max())
df_valuation.tail(15)

In [ ]:
df_valuation.columns

# 指数日频数据

数据目录：`D:\database\index_data_daily`（由 `获得指数日频数据.py` 写入，hive 分区 `year=*/month=*/merged.parquet`）。

默认关注：`000001.SH`（上证指数）、`399001.SZ`（深证成指）。

In [35]:
INDEX_BASE_PATH = r"D:\database\index_data_daily"
INDEX_VIEW_NAME = "index_day_merged"

import duckdb

con = duckdb.connect()   # 初始化 DuckDB 连接
con.execute(f"""
CREATE OR REPLACE VIEW {INDEX_VIEW_NAME} AS
SELECT *
FROM read_parquet('{INDEX_BASE_PATH}/year=*/month=*/merged.parquet', hive_partitioning=1, union_by_name=True)
""")

print("视图创建完成：", INDEX_VIEW_NAME)
print("数据路径：", INDEX_BASE_PATH)

# 可选：查看已有 year/month 分区
# con.execute(f"SELECT DISTINCT year, month FROM {INDEX_VIEW_NAME} ORDER BY year, month").df()

视图创建完成： index_day_merged
数据路径： D:\database\index_data_daily


In [36]:
# 指数池概览：有哪些代码、各自条数与日期范围
con.execute(f"""
SELECT
    UPPER(TRIM(CAST(htsc_code AS VARCHAR))) AS htsc_code,
    COUNT(*) AS row_count,
    MIN(CAST(time AS DATE)) AS first_date,
    MAX(CAST(time AS DATE)) AS last_date
FROM {INDEX_VIEW_NAME}
GROUP BY 1
ORDER BY htsc_code
""").df()

,htsc_code,row_count,first_date,last_date
0,000001.SH,3259,2013-01-04,2026-06-11
1,399001.SZ,3992,2010-01-04,2026-06-11


In [37]:
# 按指数代码查询日频（改 target_codes / end_date 即可）
target_codes = ["000001.SH", "399001.SZ"]
end_date = "2026-05-20"  # 右端日期，左端由数据自然覆盖

codes_str = ", ".join([f"'{code}'" for code in target_codes])
df_index = con.execute(f"""
SELECT *
FROM {INDEX_VIEW_NAME}
WHERE htsc_code IN ({codes_str})
  AND CAST(time AS DATE) <= DATE '{end_date}'
ORDER BY htsc_code, time
""").df()

df_index["time"] = pd.to_datetime(df_index["time"]).dt.strftime("%Y-%m-%d")
df_index.tail(10)

,htsc_code,time,security_id,frequency,open,close,high,low,num_trades,volume,value,exchange,security_type,month,year
7209,399001.SZ,2026-05-07,399001,daily,15541.2962,15641.8855,15652.5820,15439.4844,NaN,8.444899e+10,1.784856e+12,DefaultSecurityIDSource,index,05,2026
7210,399001.SZ,2026-05-08,399001,daily,15520.3808,15563.7971,15639.3878,15466.8377,NaN,8.262185e+10,1.716869e+12,DefaultSecurityIDSource,index,05,2026
7211,399001.SZ,2026-05-11,399001,daily,15697.4401,15899.3020,15921.6522,15622.3320,NaN,9.116349e+10,1.954495e+12,DefaultSecurityIDSource,index,05,2026
7212,399001.SZ,2026-05-12,399001,daily,15952.7913,15824.9211,15952.7913,15711.2706,NaN,8.478171e+10,1.777384e+12,DefaultSecurityIDSource,index,05,2026
7213,399001.SZ,2026-05-13,399001,daily,15713.5418,16089.7491,16100.4560,15713.5418,NaN,8.273414e+10,1.792550e+12,DefaultSecurityIDSource,index,05,2026
7214,399001.SZ,2026-05-14,399001,daily,16202.2151,15745.7360,16207.7519,15745.7360,NaN,8.735255e+10,1.864137e+12,DefaultSecurityIDSource,index,05,2026
7215,399001.SZ,2026-05-15,399001,daily,15753.1920,15561.3720,15855.5536,15447.4004,NaN,8.496956e+10,1.825236e+12,DefaultSecurityIDSource,index,05,2026
7216,399001.SZ,2026-05-18,399001,daily,15465.3016,15530.2295,15658.1113,15419.0380,NaN,7.383056e+10,1.578896e+12,DefaultSecurityIDSource,index,05,2026
7217,399001.SZ,2026-05-19,399001,daily,15469.3597,15569.9072,15585.0358,15221.3497,NaN,7.312165e+10,1.580036e+12,DefaultSecurityIDSource,index,05,2026
7218,399001.SZ,2026-05-20,399001,daily,15487.4267,15569.9790,15622.2137,15426.3775,NaN,7.262180e+10,1.594788e+12,DefaultSecurityIDSource,index,05,2026


In [39]:
# 单只指数全量查看（示例：上证指数）
index_code = "000001.SH"
df_index_one = con.execute(f"""
SELECT htsc_code, time, open, high, low, close, volume, value, year, month
FROM {INDEX_VIEW_NAME}
WHERE htsc_code = '{index_code}'
ORDER BY time
""").df()

print(index_code, "行数:", len(df_index_one))
if not df_index_one.empty:
    print("区间:", df_index_one["time"].min(), "~", df_index_one["time"].max())
df_index_one.tail(10)

000001.SH 行数: 3259
区间: 2013-01-04 00:00:00 ~ 2026-06-11 00:00:00


,htsc_code,time,open,high,low,close,volume,value,year,month
3249,000001.SH,2026-05-29,4110.5212,4112.9553,4055.8861,4068.5691,7.315977e+10,1.532067e+12,2026,05
3250,000001.SH,2026-06-01,4067.1578,4093.0405,4045.6898,4057.7400,6.760258e+10,1.319802e+12,2026,06
3251,000001.SH,2026-06-02,4061.4629,4089.5747,4032.5826,4075.1016,6.422447e+10,1.280966e+12,2026,06
3252,000001.SH,2026-06-03,4068.3408,4107.0458,4059.9067,4083.9740,6.684526e+10,1.429756e+12,2026,06
3253,000001.SH,2026-06-04,4053.6702,4080.7178,4043.4331,4057.7811,6.211948e+10,1.274652e+12,2026,06
3254,000001.SH,2026-06-05,4044.8292,4078.9317,4015.0629,4027.7362,6.629186e+10,1.363888e+12,2026,06
3255,000001.SH,2026-06-08,3938.7093,4007.4930,3927.8527,3959.3378,6.592936e+10,1.267277e+12,2026,06
3256,000001.SH,2026-06-09,3977.5394,4010.8715,3955.9080,4010.0307,5.765701e+10,1.173089e+12,2026,06
3257,000001.SH,2026-06-10,3985.1244,4006.3127,3963.4417,3993.2258,5.986947e+10,1.227070e+12,2026,06
3258,000001.SH,2026-06-11,3979.7057,3997.4777,3958.4371,3987.0147,5.685452e+10,1.185555e+12,2026,06


# 获得因子数据

In [ ]:
import duckdb
import pandas as pd
from typing import Optional, Iterable
import numpy as np

BASE_PATH = r"D:\database\signal_daily"   # 配置：合并后的数据根目录（只保留 year/month 分区）
VIEW_NAME = "stock_day_merged"
FACTOR = "总买入信号"
con = duckdb.connect()   # 初始化 DuckDB 连接
con.execute(f"""
CREATE OR REPLACE VIEW {VIEW_NAME} AS
SELECT *
FROM read_parquet('{BASE_PATH}/factor={FACTOR}/year=*/month=*/merged.parquet', hive_partitioning=1,union_by_name=True)
""")    # 创建视图：自动识别 year/month 分区，读取所有 merged.parquet

print("视图创建完成：", VIEW_NAME)
print("数据路径：", BASE_PATH)    

# 查看有哪些年月分区（可选）
# df_partitions = con.execute(f"SELECT DISTINCT year, month FROM {VIEW_NAME} ORDER BY year, month").df(); print("\n现有分区："); print(df_partitions)

In [ ]:
from cgi import print_arguments


target_codes = ["000905.SZ"]
codes_str = ", ".join([f"'{code}'" for code in target_codes])

df_multi = con.execute(f"SELECT * FROM {VIEW_NAME} WHERE htsc_code IN ({codes_str}) AND time >= '2026-04-01 00:00:00'  ORDER BY htsc_code, time").df()
# df_multi[['time','抄底总分']] #and df_multi['time']>'2026-04-01 16:29:18'

# 这里需要固定时间格式，方便后续的统一时间轴计算
df_multi['time'] = pd.to_datetime(df_multi['time']).dt.strftime('%Y-%m-%d')

df_multi.tail(10)


# df_multi.columns



# 获得交易数据

In [ ]:
import duckdb
import pandas as pd
from typing import Optional, Iterable
import numpy as np

BASE_PATH = r"D:\database\bs_dialy"   # 配置：合并后的数据根目录（只保留 year/month 分区）
VIEW_NAME = "stock_day_merged"

con = duckdb.connect()   # 初始化 DuckDB 连接
con.execute(f"""
CREATE OR REPLACE VIEW {VIEW_NAME} AS
SELECT *
FROM read_parquet('{BASE_PATH}/position_log_20260424_2244.parquet', hive_partitioning=1)
""")    # 创建视图：自动识别 year/month 分区，读取所有 merged.parquet

print("视图创建完成：", VIEW_NAME)
print("数据路径：", BASE_PATH)    

# 查看有哪些年月分区（可选）
# df_partitions = con.execute(f"SELECT DISTINCT year, month FROM {VIEW_NAME} ORDER BY year, month").df(); print("\n现有分区："); print(df_partitions)

In [ ]:
from cgi import print_arguments


target_codes = ["600519.SH"]
codes_str = ", ".join([f"'{code}'" for code in target_codes])

df_multi = con.execute(f"SELECT * FROM {VIEW_NAME}").df()
# df_multi[['time','抄底总分']] #and df_multi['time']>'2026-04-01 16:29:18'

# 这里需要固定时间格式，方便后续的统一时间轴计算
# df_multi['time'] = pd.to_datetime(df_multi['time']).dt.strftime('%Y-%m-%d')

df_multi#.head(10)

# df_multi.columns


In [ ]:
df_multi.columns

# 获得分钟频率数据

In [30]:
import duckdb
import pandas as pd
from pathlib import Path
from typing import Optional, Iterable
import numpy as np

MIN_BASE_PATH = r"D:\database\stock_basic_data_mins"   # ?????????year/month ???
MIN_VIEW_NAME = "stock_min_merged"

min_base = Path(MIN_BASE_PATH)
merged_files = sorted(min_base.glob("year=*/month=*/merged.parquet"))
if not merged_files:
    raise FileNotFoundError(f"????? merged.parquet?{MIN_BASE_PATH}")

parquet_source = str(min_base / "year=*" / "month=*" / "merged.parquet").replace("\\", "/")
source_desc = "year=*/month=*/merged.parquet"

con = duckdb.connect()   # ??? DuckDB ??
con.execute(f"""
CREATE OR REPLACE VIEW {MIN_VIEW_NAME} AS
SELECT *
FROM read_parquet('{parquet_source}', hive_partitioning=1, union_by_name=True)
""")

print("???????", MIN_VIEW_NAME)
print("?????", MIN_BASE_PATH)
print("?????", source_desc)

schema_df = con.execute(f"DESCRIBE SELECT * FROM {MIN_VIEW_NAME}").df()
print("\n?????")
print(schema_df["column_name"].tolist())

df_min_range = con.execute(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT htsc_code) AS stock_count,
  MIN(time) AS min_time,
  MAX(time) AS max_time
FROM {MIN_VIEW_NAME}
""").df()
print("\n?????")
print(df_min_range)


??????? stock_min_merged
????? D:\database\stock_basic_data_mins
????? year=*/month=*/merged.parquet

?????
['htsc_code', 'time', 'close', 'open', 'high', 'low', 'volume', 'amount', 'date', 'pre_close', 'change', 'pct_chg', '__index_level_0__', 'month', 'year']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


?????
         rows  stock_count            min_time            max_time
0  3427922530         5849 2010-01-04 09:30:00 2026-06-10 15:00:00


In [34]:
target_codes = ["000520.SZ"]
start_time = "2026-06-10 09:30:00"
end_time = "2026-06-10 15:00:00"

codes_str = ", ".join([f"'{code}'" for code in target_codes])

df_min_sample = con.execute(f"""
SELECT *
FROM {MIN_VIEW_NAME}
WHERE htsc_code IN ({codes_str})
  AND time >= TIMESTAMP '{start_time}'
  AND time <= TIMESTAMP '{end_time}'
ORDER BY htsc_code, time
LIMIT 1000
""").df()

if not df_min_sample.empty:
    df_min_sample["time"] = pd.to_datetime(df_min_sample["time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
df_min_sample.tail(10)

,htsc_code,time,close,open,high,low,volume,amount,date,pre_close,change,pct_chg,__index_level_0__,month,year
231,000520.SZ,2026-06-10 14:51:00,4.35,4.33,4.35,4.33,3406.0,1477625.0,2026-06-10,4.34,0.01,0.230415,11317,06,2026
232,000520.SZ,2026-06-10 14:52:00,4.34,4.34,4.35,4.34,1682.0,731278.0,2026-06-10,4.35,-0.01,-0.229885,11318,06,2026
233,000520.SZ,2026-06-10 14:53:00,4.35,4.35,4.35,4.34,2776.0,1207044.0,2026-06-10,4.34,0.01,0.230415,11319,06,2026
234,000520.SZ,2026-06-10 14:54:00,4.34,4.35,4.35,4.34,1566.0,680575.0,2026-06-10,4.35,-0.01,-0.229885,11320,06,2026
235,000520.SZ,2026-06-10 14:55:00,4.35,4.34,4.35,4.34,1766.0,767885.0,2026-06-10,4.34,0.01,0.230415,11321,06,2026
236,000520.SZ,2026-06-10 14:56:00,4.34,4.34,4.35,4.34,2923.0,1270420.0,2026-06-10,4.35,-0.01,-0.229885,11322,06,2026
237,000520.SZ,2026-06-10 14:57:00,4.34,4.34,4.35,4.34,6763.0,2938118.0,2026-06-10,4.34,0.00,0.000000,11323,06,2026
238,000520.SZ,2026-06-10 14:58:00,4.34,4.34,4.34,4.34,69.0,29946.0,2026-06-10,4.34,0.00,0.000000,11324,06,2026
239,000520.SZ,2026-06-10 14:59:00,4.34,4.34,4.34,4.34,0.0,0.0,2026-06-10,4.34,0.00,0.000000,11325,06,2026
240,000520.SZ,2026-06-10 15:00:00,4.31,4.31,4.31,4.31,11395.0,4911245.0,2026-06-10,4.34,-0.03,-0.691244,11326,06,2026


# 获得 复权因子数据

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

ADJ_BASE_PATH = r"D:\database\stock_adj_daily"
VIEW_NAME = "stock_adj_day_merged"

con = duckdb.connect()
p = Path(ADJ_BASE_PATH)
seg = p / "adj_factor_segments.parquet"
merged_glob = str(p / "year=*" / "month=*" / "merged.parquet").replace("\\", "/")
seg_s = str(seg).replace("\\", "/")

if seg.is_file():
    con.execute(f"""
CREATE OR REPLACE VIEW {VIEW_NAME} AS
SELECT *
FROM read_parquet('{seg_s}')
""")
    print("复权因子数据源：单文件 adj_factor_segments.parquet（与 获得股票日频复权因子.py 默认落盘一致）")
else:
    con.execute(f"""
CREATE OR REPLACE VIEW {VIEW_NAME} AS
SELECT *
FROM read_parquet('{merged_glob}', hive_partitioning=1, union_by_name=True)
""")
    print("复权因子数据源：year/month 分区 merged.parquet")

print("视图创建完成：", VIEW_NAME)
print("数据路径：", ADJ_BASE_PATH)


In [ ]:
target_codes = ["002612.SZ"]

# 全量获取：整表读入内存（数据量极大时可在上一 cell 改视图/SQL 做聚合检查）
df_multi = con.execute(f"SELECT * FROM {VIEW_NAME}").df()
print("总行数:", len(df_multi))
print("列名:", list(df_multi.columns))

if "time" in df_multi.columns:
    tser = pd.to_datetime(df_multi["time"], errors="coerce")
    print("time 范围:", tser.min(), "~", tser.max())
    df_multi["time"] = tser.dt.strftime("%Y-%m-%d")

if "begin_date" in df_multi.columns and "end_date" in df_multi.columns:
    b = pd.to_datetime(df_multi["begin_date"], errors="coerce")
    e = pd.to_datetime(df_multi["end_date"], errors="coerce")
    print("分段 begin_date ~ end_date:", b.min(), "~", e.max())

if "htsc_code" in df_multi.columns:
    print("不重复 htsc_code 数:", df_multi["htsc_code"].nunique())

# 模仿日频 cell：示例代码看尾部（全量仍在 df_multi）
if "htsc_code" in df_multi.columns:
    sub = df_multi[df_multi["htsc_code"].isin(target_codes)]
    if "time" in sub.columns:
        sub = sub.sort_values(["htsc_code", "time"])
    elif "begin_date" in sub.columns:
        sub = sub.sort_values(["htsc_code", "begin_date", "end_date"])
    sub.tail(10)
else:
    df_multi.tail(10)


In [ ]:
df_multi#[2000:2050]#.columns
df_multi[df_multi['htsc_code'] == '600089.SH']